# Residual Stream Congestion & Lost-in-the-Middle

**GPU-accelerated experiment pipeline**

This notebook runs the full 4-stage experiment pipeline on GPU (T4/V100/A100).

### What this notebook does:
1. **Stage 1**: ERB metric validation + initial congestion heatmap (Pythia-2.8B, 2K ctx)
2. **Stage 2**: Systematic congestion mapping across models & tasks
3. **Stage 3**: Activation Patching + Information Highway training
4. **Stage 4**: Ablation studies (ε sensitivity, highway width)

### Models supported:
- Pythia-2.8B (default, ~5.6 GB VRAM)
- Pythia-6.9B (~14 GB, requires 8-bit quantization)
- LLaMA-7B (~14 GB, requires 8-bit quantization)

### Tasks supported:
- Needle-in-a-Haystack (NiH)
- Multi-document QA (MultiDoc QA)

---

## 1. Environment Setup

In [3]:
# Install dependencies with CUDA support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q -U
!pip install transformers>=4.40.0 numpy scipy matplotlib seaborn tqdm pyyaml einops datasets accelerate -q

# Install bitsandbytes for 8-bit/4-bit quantization (required for Stage 3+4, large models)
!pip install -U bitsandbytes>=0.46.1 -q

import torch, os, sys, json, yaml
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Verify GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_props = torch.cuda.get_device_properties(0)
    gpu_mem = gpu_props.total_memory / 1e9  # Note: total_memory, not total_mem
    print(f'GPU detected: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
    device = 'cuda'
else:
    print('WARNING: No GPU detected! Running on CPU will be very slow.')
    device = 'cpu'

print(f'PyTorch version: {torch.__version__}')
print(f'Python version: {sys.version}')

# Verify bitsandbytes
try:
    import bitsandbytes as bnb
    print(f'bitsandbytes version: {bnb.__version__}')
except Exception as e:
    print(f'bitsandbytes import error: {e}')
    print('Stage 3-4 will need load_in_4bit=False or manual install.')


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
zsh:1: 4.40.0 not found
zsh:1: 0.46.1 not found
GPU detected: Tesla T4 (15.6 GB VRAM)
PyTorch version: 2.10.0+cu128
Python version: 3.11.1 (main, Feb  6 2026, 08:59:00) [GCC 13.3.0]
bitsandbytes version: 0.49.2


In [ ]:
# Create project directories
os.makedirs('experiments', exist_ok=True)
os.makedirs('figures', exist_ok=True)
print('Directories ready')

In [4]:
# Configuration
# Edit this cell to change models, context lengths, or task types

config = {
    'models': [
        {'name': 'pythia-2.8b', 'hf_path': 'EleutherAI/pythia-2.8b',
         'max_length': 2048, 'model_family': 'gpt_neox'},
# To enable larger models (requires 8-bit quantization), uncomment:
        {'name': 'pythia-6.9b', 'hf_path': 'EleutherAI/pythia-6.9b',
         'max_length': 2048, 'model_family': 'gpt_neox'},
        {'name': 'llama-7b', 'hf_path': 'huggyllama/llama-7b',
         'max_length': 2048, 'model_family': 'llama'},
    ],
    'optimization': {
        'load_in_8bit': True,      # Set True for 6.9B/7B models
        'load_in_4bit': False,
        'device_map': 'auto',
        'gradient_checkpointing': False,
    },
    'context_lengths': [1024, 2048],
    'erb': {'epsilon': 1e-8, 'norm': 'l2'},
    'nih': {
        'num_docs_list': [5, 10],
        'examples_per_setting': 50,
        'needle_template': 'The {key} of {obj} is {value}.',
        'distractor_template': 'The capital of {country} is {city}.',
        'answer_format': 'The special value is: {value}',
    },
    'multidoc_qa': {
        'num_docs_list': [10, 20, 30],
        'examples_per_setting': 50,
        'answer_values': ['A', 'B', 'C', 'D', 'E', '1', '2', '3', '4', '5'],
    },
    'patching': {
        'bottleneck_layers': 5,
        'control_patch_layers': 2,
    },
    'highway': {
        'width_factor': 2,
        'num_layers': 3,
        'learning_rate': 0.001,
        'epochs': 5,
        'batch_size': 8,
        'training_examples': 100,
        'eval_examples': 50,
    },
    'experiment': {
        'seeds': [42, 43, 44],
        'device': device,
        'dtype': 'float32',
        'output_dir': 'experiments',
        'task_types': ['nih', 'multidoc_qa'],
    },
}

# Save config for module use
with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('Config saved. Models:', [m['name'] for m in config['models']])
print('Task types:', config['experiment']['task_types'])
print('Context lengths:', config['context_lengths'])

Config saved. Models: ['pythia-2.8b', 'pythia-6.9b', 'llama-7b']
Task types: ['nih', 'multidoc_qa']
Context lengths: [1024, 2048]


## 2. Core Modules

These cells define the Python modules used by the experiment pipeline.

In [ ]:
"""
ERB metric — defined in notebook AND written to erb_metric.py for other modules.
"""
import torch
import numpy as np
from typing import Dict, Optional, Tuple

class ERBMetric:
    """ERB(l,p) = ||x_l,p|| / (||Δx_l,p|| + ε)"""
    def __init__(self, epsilon: float = 1e-8, norm_type: str = 'l2'):
        self.epsilon = epsilon
        self.norm_type = norm_type
    def _norm(self, x):
        if self.norm_type == 'l2': return torch.norm(x, dim=-1, p=2)
        elif self.norm_type == 'l1': return torch.norm(x, dim=-1, p=1)
        else: return torch.ones(x.shape[:-1], device=x.device)
    def compute(self, layer_input, layer_output):
        delta = layer_output - layer_input
        return self._norm(layer_output) / (self._norm(delta) + self.epsilon)
    def compute_all(self, activations, num_layers):
        first_key = [k for k in activations if k.endswith('_input')][0]
        seq_len = activations[first_key].shape[1]
        erb_matrix = torch.zeros(num_layers, seq_len)
        delta_norms = torch.zeros(num_layers, seq_len)
        for layer_idx in range(num_layers):
            key_in, key_out = f'layer_{layer_idx}_input', f'layer_{layer_idx}'
            if key_in not in activations or key_out not in activations: continue
            li, lo = activations[key_in], activations[key_out]
            if li.ndim == 3: li, lo = li.mean(dim=0), lo.mean(dim=0)
            erb_matrix[layer_idx] = self.compute(li, lo)
            delta_norms[layer_idx] = self._norm(lo - li)
        return erb_matrix, delta_norms
    @staticmethod
    def compute_position_profile(erb_matrix):
        return erb_matrix.mean(dim=0).numpy()
    @staticmethod
    def compute_bottleneck_regions(erb_matrix, top_k=5):
        flat_vals = erb_matrix.flatten()
        _, indices = torch.topk(flat_vals, k=min(top_k*10, len(flat_vals)), largest=False)
        bottlenecks, seen_pos = [], set()
        for idx in indices:
            layer, pos = idx.item() // erb_matrix.shape[1], idx.item() % erb_matrix.shape[1]
            pg = pos // max(erb_matrix.shape[1] // 10, 1)
            if pg not in seen_pos:
                seen_pos.add(pg)
                bottlenecks.append((layer, pos, erb_matrix[layer, pos].item()))
            if len(bottlenecks) >= top_k: break
        return bottlenecks
    @staticmethod
    def spearman_correlation(erb_profile, accuracy_by_position):
        from scipy.stats import spearmanr
        m = min(len(erb_profile), len(accuracy_by_position))
        return spearmanr(erb_profile[:m], accuracy_by_position[:m])

# ALSO write to disk so congestion_mapper.py can import it
with open('erb_metric.py', 'w') as f:
    f.write('''import torch
import numpy as np

class ERBMetric:
    def __init__(self, epsilon=1e-8, norm_type='l2'):
        self.epsilon = epsilon; self.norm_type = norm_type
    def _norm(self, x):
        if self.norm_type == 'l2': return torch.norm(x, dim=-1, p=2)
        elif self.norm_type == 'l1': return torch.norm(x, dim=-1, p=1)
        else: return torch.ones(x.shape[:-1], device=x.device)
    def compute(self, li, lo):
        return self._norm(lo) / (self._norm(lo - li) + self.epsilon)
    def compute_all(self, activations, num_layers):
        fk = [k for k in activations if k.endswith('_input')][0]
        S = activations[fk].shape[1]; E = torch.zeros(num_layers, S)
        for i in range(num_layers):
            ki, ko = f'layer_{i}_input', f'layer_{i}'
            if ki not in activations or ko not in activations: continue
            li, lo = activations[ki], activations[ko]
            if li.ndim == 3: li, lo = li.mean(0), lo.mean(0)
            E[i] = self.compute(li, lo)
        return E, torch.zeros(num_layers, S)
    @staticmethod
    def compute_position_profile(m): return m.mean(0).numpy()
    @staticmethod
    def compute_bottleneck_regions(m, k=5):
        _, idx = torch.topk(m.flatten(), k=min(k*10, len(m.flatten())), largest=False)
        b, seen = [], set()
        for i in idx:
            l, p = i.item() // m.shape[1], i.item() % m.shape[1]
            pg = p // max(m.shape[1]//10, 1)
            if pg not in seen: seen.add(pg); b.append((l, p, m[l,p].item()))
            if len(b) >= k: break
        return b
    @staticmethod
    def spearman_correlation(e, a):
        from scipy.stats import spearmanr
        m = min(len(e), len(a)); return spearmanr(e[:m], a[:m])
''')
print('ERBMetric defined in notebook + written to erb_metric.py')

In [ ]:
"""
ModelLoader + ActivationCache — defined in notebook AND written to model_loader.py.
"""
import torch, torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Dict, List, Optional, Tuple

class ActivationCache:
    def __init__(self): self.reset()
    def reset(self):
        self.activations: Dict[str, torch.Tensor] = {}
        self.hooks = []
    def _make_hook(self, name, store_input=False):
        def hook(module, input, output):
            if store_input:
                self.activations[f'{name}_input'] = input[0].detach().cpu()
            self.activations[name] = output[0].detach().cpu()
        return hook
    def clear(self):
        self.reset()
        for h in self.hooks: h.remove()
        self.hooks = []

class ModelLoader:
    def __init__(self, model_name: str, device: str = 'cuda', dtype: str = 'float16',
                 cache: Optional[ActivationCache] = None,
                 load_in_8bit: bool = False, load_in_4bit: bool = False,
                 device_map: Optional[str] = None):
        self.model_name, self.cache = model_name, cache or ActivationCache()
        self.device = device if torch.cuda.is_available() and device == 'cuda' else 'cpu'
        print(f'[ModelLoader] Loading {model_name}...')
        if load_in_8bit or load_in_4bit:
            self.dtype = None
        else:
            self.dtype = getattr(torch, dtype) if dtype != 'float16' or torch.cuda.is_available() else torch.float32
        if load_in_8bit:
            from transformers import BitsAndBytesConfig
            self.model = AutoModelForCausalLM.from_pretrained(model_name,
                quantization_config=BitsAndBytesConfig(load_in_8bit=True),
                device_map='auto', trust_remote_code=True)
        elif load_in_4bit:
            from transformers import BitsAndBytesConfig
            self.model = AutoModelForCausalLM.from_pretrained(model_name,
                quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
                device_map='auto', trust_remote_code=True)
        else:
            dm = device_map or ('auto' if self.device == 'cuda' else None)
            self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=self.dtype,
                device_map=dm, trust_remote_code=True)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token or '[PAD]'
            if self.tokenizer.pad_token == '[PAD]':
                self.tokenizer.add_special_tokens({'pad_token': '[PAD]'})
                self.model.resize_token_embeddings(len(self.tokenizer))
        if self.tokenizer.model_max_length > 1_000_000:
            self.tokenizer.model_max_length = 2048
        self.model.eval()
        cfg = self.model.config
        self.num_layers = next(getattr(cfg, a) for a in ['num_hidden_layers','n_layer','num_layers'] if hasattr(cfg, a))
        self.hidden_size = cfg.hidden_size
        self.head_dim = getattr(cfg, 'head_dim', cfg.hidden_size // cfg.num_attention_heads)
        self.is_quantized = load_in_8bit or load_in_4bit
        print(f'[ModelLoader] Ready: {self.num_layers} layers, {self.hidden_size} hidden, quantized={self.is_quantized}')
    def _get_transformer_layers(self):
        if hasattr(self.model, 'gpt_neox'): return self.model.gpt_neox.layers
        elif hasattr(self.model, 'transformer') and hasattr(self.model.transformer, 'h'):
            return self.model.transformer.h
        elif hasattr(self.model.model, 'layers'): return self.model.model.layers
        raise AttributeError(f'Cannot locate layers for {self.model_name}')
    def register_activation_hooks(self, layer_indices=None):
        self.cache.clear()
        layers = self._get_transformer_layers()
        if layer_indices is None: layer_indices = list(range(len(layers)))
        for idx in layer_indices:
            if idx >= len(layers): continue
            h = layers[idx].register_forward_hook(self.cache._make_hook(f'layer_{idx}', store_input=True))
            self.cache.hooks.append(h)
    @torch.no_grad()
    def forward_with_cache(self, input_ids, layer_indices=None):
        self.cache.reset()
        self.register_activation_hooks(layer_indices)
        outputs = self.model(input_ids)
        return outputs, self.cache
    def to(self, device):
        if not self.is_quantized: self.model = self.model.to(device); self.device = device
        return self

# ALSO write to disk so congestion_mapper.py can import
with open('model_loader.py', 'w') as f:
    f.write('''import torch, torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Dict, Optional

class ActivationCache:
    def __init__(self): self.reset()
    def reset(self):
        self.activations = {}; self.hooks = []
    def _make_hook(self, name, si=False):
        def hook(m, i, o):
            if si: self.activations[f'{name}_input'] = i[0].detach().cpu()
            self.activations[name] = o[0].detach().cpu()
        return hook
    def clear(self):
        self.reset()
        for h in self.hooks: h.remove(); self.hooks = []

class ModelLoader:
    def __init__(self, mn, device='cuda', dtype='float16', cache=None, load_in_8bit=False, load_in_4bit=False, device_map=None):
        self.model_name, self.cache = mn, cache or ActivationCache()
        self.device = device if torch.cuda.is_available() and device == 'cuda' else 'cpu'
        print(f'[ModelLoader] Loading {mn}...')
        if load_in_8bit or load_in_4bit: self.dtype = None
        else: self.dtype = getattr(torch, dtype) if dtype != 'float16' or torch.cuda.is_available() else torch.float32
        if load_in_8bit:
            from transformers import BitsAndBytesConfig
            self.model = AutoModelForCausalLM.from_pretrained(mn, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map='auto', trust_remote_code=True)
        elif load_in_4bit:
            from transformers import BitsAndBytesConfig
            self.model = AutoModelForCausalLM.from_pretrained(mn, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16), device_map='auto', trust_remote_code=True)
        else:
            dm = device_map or ('auto' if self.device == 'cuda' else None)
            self.model = AutoModelForCausalLM.from_pretrained(mn, torch_dtype=self.dtype, device_map=dm, trust_remote_code=True)
        self.tokenizer = AutoTokenizer.from_pretrained(mn, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token or '[PAD]'
            if self.tokenizer.pad_token == '[PAD]':
                self.tokenizer.add_special_tokens({'pad_token': '[PAD]'}); self.model.resize_token_embeddings(len(self.tokenizer))
        if self.tokenizer.model_max_length > 1_000_000: self.tokenizer.model_max_length = 2048
        self.model.eval()
        cfg = self.model.config
        self.num_layers = next(getattr(cfg, a) for a in ['num_hidden_layers','n_layer','num_layers'] if hasattr(cfg,a))
        self.hidden_size = cfg.hidden_size
        self.head_dim = getattr(cfg, 'head_dim', cfg.hidden_size // cfg.num_attention_heads)
        self.is_quantized = load_in_8bit or load_in_4bit
        print(f'[ModelLoader] Ready: {self.num_layers} layers, {self.hidden_size} hidden')
    def _get_transformer_layers(self):
        if hasattr(self.model, 'gpt_neox'): return self.model.gpt_neox.layers
        elif hasattr(self.model, 'transformer') and hasattr(self.model.transformer, 'h'): return self.model.transformer.h
        elif hasattr(self.model.model, 'layers'): return self.model.model.layers
        raise AttributeError(f'Cannot locate layers')
    def register_activation_hooks(self, li=None):
        self.cache.clear(); layers = self._get_transformer_layers()
        if li is None: li = list(range(len(layers)))
        for idx in li:
            if idx >= len(layers): continue
            self.cache.hooks.append(layers[idx].register_forward_hook(self.cache._make_hook(f'layer_{idx}', True)))
    @torch.no_grad()
    def forward_with_cache(self, ids, li=None):
        self.cache.reset(); self.register_activation_hooks(li); return self.model(ids), self.cache
    def to(self, d):
        if not self.is_quantized: self.model = self.model.to(d); self.device = d
        return self
''')
print('ModelLoader + ActivationCache defined in notebook + written to model_loader.py')

In [ ]:
"""
NiH generator — defined in notebook AND written to needle_haystack.py.
"""
import random, torch, numpy as np
from typing import List, Dict

COUNTRIES_CITIES = [('France','Paris'),('Germany','Berlin'),('Italy','Rome'),
    ('Spain','Madrid'),('UK','London'),('Japan','Tokyo'),('China','Beijing'),
    ('India','New Delhi'),('Brazil','Brasilia'),('Canada','Ottawa'),
    ('Australia','Canberra'),('Russia','Moscow'),('Mexico','Mexico City'),('Egypt','Cairo')]
NEEDLE_VALUES = ['A','B','C','D','E','1','2','3','4','5']

class NeedleHaystackGenerator:
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def generate_dataset(self, num_docs_list, context_length=2048,
                          examples_per_setting=50, tokenizer=None, seed=42):
        self.rng = random.Random(seed)
        dataset, objects = [], ['planet','star','color','element','animal','fruit']
        for num_docs in num_docs_list:
            for pos in range(num_docs + 1):
                for _ in range(max(1, examples_per_setting // (num_docs + 1))):
                    obj = self.rng.choice(objects)
                    val = self.rng.choice(NEEDLE_VALUES)
                    needle = f'The special {obj} is {val}.'
                    distractors = [f'The capital of {c} is {cy}.' for c, cy in
                        self.rng.sample(COUNTRIES_CITIES, num_docs)]
                    docs = distractors[:pos] + [needle] + distractors[pos:]
                    text = ' | '.join(f'Doc {i+1}: {d}' for i,d in enumerate(docs))
                    text += f'\nRetrieve the {obj} from the docs above.\nAnswer: '
                    dataset.append({'text': text, 'answer': val, 'obj': obj,
                        'needle_pos': pos, 'normalized_pos': pos / max(num_docs, 1),
                        'num_docs': num_docs})
                    if len(dataset) >= examples_per_setting * len(num_docs_list): break
                if len(dataset) >= examples_per_setting * len(num_docs_list): break
        print(f'[NiH] Generated {len(dataset)} examples')
        return dataset

with open('needle_haystack.py', 'w') as f:
    f.write('''import random
COUNTRIES_CITIES = [('France','Paris'),('Germany','Berlin'),('Italy','Rome'),('Spain','Madrid'),('UK','London')]
NEEDLE_VALUES = ['A','B','C','D','E','1','2','3','4','5']
class NeedleHaystackGenerator:
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def generate_dataset(self, num_docs_list, context_length=2048, examples_per_setting=50, tokenizer=None, seed=42):
        self.rng = random.Random(seed); dataset, objects = [], ['planet','star','color','element','animal','fruit']
        for num_docs in num_docs_list:
            for pos in range(num_docs + 1):
                for _ in range(max(1, examples_per_setting // (num_docs + 1))):
                    obj = self.rng.choice(objects); val = self.rng.choice(NEEDLE_VALUES)
                    needle = f'The special {obj} is {val}.'
                    distractors = [f'The capital of {c} is {cy}.' for c,cy in self.rng.sample(COUNTRIES_CITIES, num_docs)]
                    docs = distractors[:pos] + [needle] + distractors[pos:]
                    text = ' | '.join(f'Doc {i+1}: {d}' for i,d in enumerate(docs)) + f'\\nRetrieve the {obj} from the docs above.\\nAnswer: '
                    dataset.append({'text':text,'answer':val,'obj':obj,'needle_pos':pos,'normalized_pos':pos/max(num_docs,1),'num_docs':num_docs})
                    if len(dataset) >= examples_per_setting * len(num_docs_list): break
                if len(dataset) >= examples_per_setting * len(num_docs_list): break
        print(f'[NiH] Generated {len(dataset)} examples'); return dataset
''')
print('NeedleHaystackGenerator defined + written to needle_haystack.py')

In [ ]:
"""
Multi-doc QA generator — defined in notebook AND written to multidoc_qa.py.
"""
import random, torch, numpy as np
from typing import List, Dict

ENTITIES = [('Mercury','color','gray'),('Venus','color','yellow'),('Earth','color','blue'),
    ('Mars','color','red'),('Jupiter','color','orange'),('Saturn','color','golden'),
    ('Uranus','color','cyan'),('Neptune','color','blue'),('Pluto','color','brown'),
    ('Sun','type','star'),('Moon','type','satellite'),('Titan','type','moon')]
DISTRACTORS = [('Python','creator','Guido'),('Linux','creator','Torvalds'),
    ('HTTP','protocol','web'),('JSON','format','text'),('SQL','language','query'),
    ('Git','tool','version'),('Docker','tool','container'),('Redis','database','cache')]
ANSWERS = ['A','B','C','D','E','1','2','3','4','5']

class MultiDocQAGenerator:
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def generate_dataset(self, num_docs_list, context_length=2048,
                          examples_per_setting=50, tokenizer=None, seed=42):
        self.rng = random.Random(seed); dataset = []
        for num_docs in num_docs_list:
            for pos in range(num_docs):
                for _ in range(max(1, examples_per_setting // num_docs)):
                    e, attr, _ = self.rng.choice(ENTITIES); val = self.rng.choice(ANSWERS)
                    docs = []
                    for i in range(num_docs):
                        if i == pos: s = f'The {attr} of {e} is {val}.'
                        else: d,a,v = self.rng.choice(DISTRACTORS); s = f'The {a} of {d} is {v}.'
                        docs.append(f'Doc {i+1}: {s}')
                    text = '\\n'.join(docs) + f'\\n\\nAccording to the documents, what is the {attr} of {e}?\\nAnswer: '
                    dataset.append({'text': text, 'answer': val, 'attribute': attr,
                        'answer_doc_pos': pos, 'normalized_pos': pos / max(num_docs - 1, 1),
                        'num_docs': num_docs})
                    if len(dataset) >= examples_per_setting * len(num_docs_list): break
                if len(dataset) >= examples_per_setting * len(num_docs_list): break
        print(f'[MultiDocQA] Generated {len(dataset)} examples'); return dataset

with open('multidoc_qa.py', 'w') as f:
    f.write('''import random
ENTITIES = [('Mercury','color','gray'),('Venus','color','yellow'),('Earth','color','blue'),('Mars','color','red')]
DISTRACTORS = [('Python','creator','Guido'),('Linux','creator','Torvalds'),('HTTP','protocol','web'),('JSON','format','text')]
ANSWERS = ['A','B','C','D','E','1','2','3','4','5']
class MultiDocQAGenerator:
    def __init__(self, seed=42): self.rng = random.Random(seed)
    def generate_dataset(self, num_docs_list, context_length=2048, examples_per_setting=50, tokenizer=None, seed=42):
        self.rng = random.Random(seed); dataset = []
        for num_docs in num_docs_list:
            for pos in range(num_docs):
                for _ in range(max(1, examples_per_setting // num_docs)):
                    e,attr,_ = self.rng.choice(ENTITIES); val = self.rng.choice(ANSWERS)
                    docs = []
                    for i in range(num_docs):
                        if i == pos: s = f'The {attr} of {e} is {val}.'
                        else: d,a,v = self.rng.choice(DISTRACTORS); s = f'The {a} of {d} is {v}.'
                        docs.append(f'Doc {i+1}: {s}')
                    text = '\\n'.join(docs) + f'\\n\\nAccording to the documents, what is the {attr} of {e}?\\nAnswer: '
                    dataset.append({'text':text,'answer':val,'attribute':attr,'answer_doc_pos':pos,'normalized_pos':pos/max(num_docs-1,1),'num_docs':num_docs})
                    if len(dataset) >= examples_per_setting * len(num_docs_list): break
                if len(dataset) >= examples_per_setting * len(num_docs_list): break
        print(f'[MultiDocQA] Generated {len(dataset)} examples'); return dataset
''')
print('MultiDocQAGenerator defined + written to multidoc_qa.py')

In [ ]:
"""
CongestionMapper — defined in notebook AND written to congestion_mapper.py.
Uses classes from the notebook's global namespace (not module imports).
"""
import os, json, torch, numpy as np

class CongestionMapper:
    def __init__(self, config):
        self.config = config
        self.erb_metric = ERBMetric(epsilon=config['erb']['epsilon'], norm_type=config['erb']['norm'])
    def run_single_setting(self, model_name, context_length, num_docs,
                            num_examples=50, seed=42, device='cuda', task_type='nih'):
        print(f'Running: {model_name}, ctx={context_length}, docs={num_docs}, task={task_type}')
        opt = self.config.get('optimization', {})
        loader = ModelLoader(model_name, device=device, dtype=self.config['experiment']['dtype'],
            cache=ActivationCache(), load_in_8bit=opt.get('load_in_8bit', False),
            load_in_4bit=opt.get('load_in_4bit', False), device_map=opt.get('device_map', None))
        Gen = NeedleHaystackGenerator if task_type == 'nih' else MultiDocQAGenerator
        dataset = Gen(seed=seed).generate_dataset([num_docs], context_length,
            num_examples, loader.tokenizer, seed)
        erb_matrices, acc_by_pos = [], {}
        for i, ex in enumerate(dataset):
            inputs = loader.tokenizer(ex['text'], return_tensors='pt', truncation=True)
            input_ids = inputs.input_ids.to(device)
            try:
                outputs, cache = loader.forward_with_cache(input_ids)
                erb, _ = self.erb_metric.compute_all(cache.activations, loader.num_layers)
                erb_matrices.append(erb)
                log_probs = torch.log_softmax(outputs.logits[0, -1, :], dim=-1)
                ans_tokens = loader.tokenizer.encode(' ' + ex['answer'], add_special_tokens=False)
                if not ans_tokens: ans_tokens = loader.tokenizer.encode(ex['answer'], add_special_tokens=False)
                score = log_probs[ans_tokens[0]].item() + torch.tensor(len(log_probs)).float().log().item() if ans_tokens else 0.0
                pb = round(ex['normalized_pos'] * 10) / 10
                if pb not in acc_by_pos: acc_by_pos[pb] = {'scores': [], 'count': 0}
                acc_by_pos[pb]['scores'].append(score); acc_by_pos[pb]['count'] += 1
            except Exception as e:
                print(f'  Error ex {i}: {e}'); continue
        if not erb_matrices: return {'error': 'No valid ERB'}
        lengths = [m.shape[1] for m in erb_matrices]
        padded = [torch.cat([m, m[:,-1:].expand(-1, max(lengths)-m.shape[1])], dim=1) if m.shape[1] < max(lengths) else m for m in erb_matrices]
        stacked = torch.stack(padded)[:, :, :min(lengths)]
        mean_erb = stacked.mean(dim=0)
        positions = sorted(acc_by_pos.keys())
        acc_vals = np.array([np.mean(acc_by_pos[p]['scores']) or 0.0 for p in positions])
        profile = self.erb_metric.compute_position_profile(mean_erb)
        m = min(len(profile), len(acc_vals))
        rho, p_val = self.erb_metric.spearman_correlation(profile[:m], acc_vals[:m]) if m > 2 else (0.0, 1.0)
        return {'model': model_name, 'context_length': context_length, 'num_docs': num_docs,
            'seed': seed, 'task_type': task_type, 'mean_erb_npy': mean_erb.numpy().tolist(),
            'erb_profile': profile.tolist(), 'score_values': acc_vals.tolist(),
            'positions': positions, 'spearman_rho': float(rho), 'spearman_p': float(p_val),
            'bottlenecks': self.erb_metric.compute_bottleneck_regions(mean_erb), 'num_valid_examples': len(erb_matrices)}
    def run_grid(self, output_dir='experiments'):
        cfg = self.config; all_results = {}
        for mc in cfg['models']:
            for ctx in cfg['context_lengths']:
                for tt in cfg['experiment']['task_types']:
                    dl = cfg['nih']['num_docs_list'] if tt == 'nih' else cfg['multidoc_qa']['num_docs_list']
                    ne = cfg['nih']['examples_per_setting'] if tt == 'nih' else cfg['multidoc_qa']['examples_per_setting']
                    for nd in dl:
                        for s in cfg['experiment']['seeds']:
                            tag = f'_{tt}' if tt != 'nih' else ''
                            rk = f"{mc['name']}_{ctx}_{nd}_{s}{tag}"
                            try:
                                all_results[rk] = self.run_single_setting(mc['hf_path'], ctx, nd, ne, s, cfg['experiment']['device'], tt)
                            except Exception as e: print(f'[ERROR] {rk}: {e}')
        os.makedirs(output_dir, exist_ok=True)
        with open(os.path.join(output_dir, 'congestion_results.json'), 'w') as f:
            json.dump({k: {kk: vv for kk, vv in v.items() if kk not in ('mean_erb_npy',)}
                for k, v in all_results.items() if isinstance(v, dict)}, f, indent=2)
        return all_results

# Write .py version with proper imports for run_grid()
with open('congestion_mapper.py', 'w') as f:
    f.write('''import os, json, torch, numpy as np
from model_loader import ModelLoader, ActivationCache
from erb_metric import ERBMetric
from needle_haystack import NeedleHaystackGenerator
from multidoc_qa import MultiDocQAGenerator

class CongestionMapper:
    def __init__(self, config):
        self.config = config
        self.erb_metric = ERBMetric(epsilon=config['erb']['epsilon'], norm_type=config['erb']['norm'])
    def run_single_setting(self, model_name, ctx_len, num_docs, num_examples=50, seed=42, device="cuda", task_type="nih"):
        opt = self.config.get("optimization", {})
        loader = ModelLoader(model_name, device=device, dtype=self.config["experiment"]["dtype"],
            cache=ActivationCache(), load_in_8bit=opt.get("load_in_8bit", False),
            load_in_4bit=opt.get("load_in_4bit", False), device_map=opt.get("device_map", None))
        Gen = NeedleHaystackGenerator if task_type == "nih" else MultiDocQAGenerator
        dataset = Gen(seed=seed).generate_dataset([num_docs], ctx_len, num_examples, loader.tokenizer, seed)
        mats, acc = [], {}
        for i, ex in enumerate(dataset):
            ids = loader.tokenizer(ex["text"], return_tensors="pt", truncation=True).input_ids.to(device)
            try:
                o, c = loader.forward_with_cache(ids); erb, _ = self.erb_metric.compute_all(c.activations, loader.num_layers)
                mats.append(erb)
                lp = torch.log_softmax(o.logits[0, -1, :], dim=-1)
                at = loader.tokenizer.encode(" " + ex["answer"], add_special_tokens=False) or loader.tokenizer.encode(ex["answer"], add_special_tokens=False)
                sc = lp[at[0]].item() + torch.tensor(len(lp)).float().log().item() if at else 0.0
                pb = round(ex["normalized_pos"] * 10) / 10
                if pb not in acc: acc[pb] = {"scores": [], "count": 0}
                acc[pb]["scores"].append(sc); acc[pb]["count"] += 1
            except: continue
        if not mats: return {"error": "No valid ERB"}
        L = [m.shape[1] for m in mats]
        pad = [torch.cat([m, m[:,-1:].expand(-1, max(L)-m.shape[1])], dim=1) if m.shape[1] < max(L) else m for m in mats]
        mean = torch.stack(pad)[:, :, :min(L)].mean(0)
        pos = sorted(acc.keys()); av = np.array([np.mean(acc[p]["scores"]) or 0.0 for p in pos])
        prof = ERBMetric.compute_position_profile(mean)
        m = min(len(prof), len(av))
        r, pv = ERBMetric.spearman_correlation(prof[:m], av[:m]) if m > 2 else (0.0, 1.0)
        return {"model": model_name, "context_length": ctx_len, "num_docs": num_docs, "seed": seed,
            "task_type": task_type, "mean_erb_npy": mean.numpy().tolist(), "erb_profile": prof.tolist(),
            "score_values": av.tolist(), "positions": pos, "spearman_rho": float(r), "spearman_p": float(pv),
            "bottlenecks": ERBMetric.compute_bottleneck_regions(mean), "num_valid_examples": len(mats)}
    def run_grid(self, output_dir="experiments"):
        cfg, all_r = self.config, {}
        for mc in cfg["models"]:
            for ctx in cfg["context_lengths"]:
                for tt in cfg["experiment"]["task_types"]:
                    dl = cfg["nih"]["num_docs_list"] if tt == "nih" else cfg["multidoc_qa"]["num_docs_list"]
                    ne = cfg["nih"]["examples_per_setting"] if tt == "nih" else cfg["multidoc_qa"]["examples_per_setting"]
                    for nd in dl:
                        for s in cfg["experiment"]["seeds"]:
                            rk = f"{mc['name']}_{ctx}_{nd}_{s}" + (f"_{tt}" if tt != "nih" else "")
                            try: all_r[rk] = self.run_single_setting(mc["hf_path"], ctx, nd, ne, s, cfg["experiment"]["device"], tt)
                            except Exception as e: print(f"[ERROR] {rk}: {e}")
        os.makedirs(output_dir, exist_ok=True)
        with open(os.path.join(output_dir, "congestion_results.json"), "w") as f:
            json.dump({k: {kk: vv for kk, vv in v.items() if kk not in ("mean_erb_npy",)} for k, v in all_r.items() if isinstance(v, dict)}, f, indent=2)
        return all_r
''')
print('CongestionMapper defined + written to congestion_mapper.py')

In [ ]:
# All modules are defined directly in the notebook namespace above.
# Verify they're available:
for name in ['ERBMetric', 'ModelLoader', 'ActivationCache', 'NeedleHaystackGenerator', 'MultiDocQAGenerator', 'CongestionMapper']:
    assert name in dir(), f'{name} not found!'
print('All module classes verified in notebook namespace!')

# For Stage 2 run_grid() which needs module imports,
# also do the standard import so congestion_mapper.py works as a module:
from congestion_mapper import CongestionMapper as CMCached

## 3. Stage 1: Initial Implementation

Validate ERB metric and generate first congestion heatmap.

In [ ]:
print('=' * 60)
print('STAGE 1: Initial Implementation (Pythia-2.8B, 2K ctx, NiH)')
print('=' * 60)

mapper = CongestionMapper(config)
result = mapper.run_single_setting(
    model_name=config['models'][0]['hf_path'],
    context_length=2048, num_docs=5,
    num_examples=20, seed=42, device=device,
    task_type='nih')

if 'error' in result:
    print(f'FAILED: {result["error"]}')
else:
    print(f'\nResults:')
    print(f'  Valid examples: {result["num_valid_examples"]}')
    print(f'  Spearman rho: {result["spearman_rho"]:.3f}')
    print(f'  Spearman p: {result["spearman_p"]:.4f}')
    print(f'  Top bottlenecks: {result["bottlenecks"][:3]}')

    # Plot heatmap
    plt.figure(figsize=(10, 6))
    erb = np.array(result['mean_erb_npy'])
    plt.imshow(erb, aspect='auto', cmap='RdYlBu_r', origin='lower')
    plt.colorbar(label='ERB (lower = more congested)')
    plt.xlabel('Token Position')
    plt.ylabel('Layer Index')
    plt.title(f'ERB Heatmap: Pythia-2.8B, ctx=2048')
    plt.tight_layout()
    plt.savefig('experiments/stage1_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Stage 1 complete!')

## 4. Stage 2: Systematic Congestion Mapping

Run full grid: models × context lengths × task types × seeds.

In [ ]:
print('=' * 60)
print('STAGE 2: Systematic Congestion Mapping')
print('=' * 60)
print(f'Task types: {config["experiment"]["task_types"]}')
print(f'Models: {[m["name"] for m in config["models"]]}')

all_results = mapper.run_grid(output_dir='experiments')

# Summary
spearman_values = []
for k, v in all_results.items():
    if isinstance(v, dict) and 'spearman_rho' in v:
        spearman_values.append(v['spearman_rho'])
        print(f'  {k}: rho={v["spearman_rho"]:.3f}')

if spearman_values:
    print(f'\nMean Spearman rho: {np.mean(spearman_values):.3f} ± {np.std(spearman_values):.3f}')
print('Stage 2 complete!')

In [ ]:
import json

output_file_path = os.path.join(config['experiment']['output_dir'], 'Stage2_all_results.json')
with open(output_file_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'Stage 2 All experiment results saved to: {output_file_path}')

## 5. Stage 3: Activation Patching + Information Highway

In [ ]:
# === Stage 3: Completely self-contained ===
# Part A: Activation Patching (causal validation via corrupted-input)
# Part B: Information Highway Training + Accuracy Evaluation
#
# Bug fixes from previous run:
#   1. Patching: Switched from easy->hard cross-text patching (positions mismatch)
#      to corrupted-input patching (same text, needle removed & patched back).
#   2. Highway: Switched from zero-init -> Xavier init; masked context tokens
#      in labels to prevent fp16 overflow.

import torch, torch.nn as nn, os, sys, json, random
import numpy as np

print('=' * 60)
print('STAGE 3: Causal Validation + Mitigation')
print('=' * 60)

# ===== Model config =====
MODEL_NAME = 'EleutherAI/pythia-2.8b'

# ===== Check bitsandbytes =====
try:
    import bitsandbytes as bnb
    BNB_AVAILABLE = True
    print(f'bitsandbytes v{bnb.__version__} detected')
except ImportError:
    BNB_AVAILABLE = False
    print('bitsandbytes NOT detected, loading full precision')

# ===== Load model =====
from transformers import AutoModelForCausalLM, AutoTokenizer

if BNB_AVAILABLE:
    from transformers import BitsAndBytesConfig
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=qcfg,
        device_map='auto', trust_remote_code=True)
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
        torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
        device_map='auto' if device == 'cuda' else None,
        trust_remote_code=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model.eval()

cfg = model.config
num_layers = next(getattr(cfg, a) for a in ['num_hidden_layers','n_layer','num_layers'] if hasattr(cfg,a))
hidden_size = cfg.hidden_size
head_dim = getattr(cfg, 'head_dim', hidden_size // cfg.num_attention_heads)
model_device = next(model.parameters()).device
model_dtype = next(model.parameters()).dtype
backbone_params = sum(p.numel() for p in model.parameters())
print(f'Model: {num_layers} layers, {hidden_size} hidden, {backbone_params:,} params')
print(f'Device: {model_device}, dtype: {model_dtype}')


# ============================================================
# SHARED UTILITIES
# ============================================================

_countries = [('France','Paris'),('Germany','Berlin'),('Italy','Rome'),('Spain','Madrid'),
              ('UK','London'),('Japan','Tokyo'),('China','Beijing'),('India','New Delhi'),
              ('Australia','Canberra'),('Russia','Moscow'),('Brazil','Brasilia'),
              ('Canada','Ottawa'),('Egypt','Cairo'),('Mexico','Mexico City')]
_objects = ['planet','star','color','element','animal','fruit']
_answers = ['A','B','C','D','E']

def score_answer(model, tokenizer, text, answer, device):
    """Log-probability of the answer token given text (higher = better)."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True).to(device)
    with torch.no_grad():
        logits = model(inputs.input_ids).logits
    log_probs = torch.log_softmax(logits[0, -1, :], dim=-1)
    ans_tokens = tokenizer.encode(' ' + answer, add_special_tokens=False)
    if not ans_tokens:
        ans_tokens = tokenizer.encode(answer, add_special_tokens=False)
    return log_probs[ans_tokens[0]].item() if ans_tokens else 0.0

def _get_layers(model):
    """Get transformer layer list for any architecture."""
    if hasattr(model, 'gpt_neox'): return model.gpt_neox.layers
    if hasattr(model, 'transformer') and hasattr(model.transformer, 'h'): return model.transformer.h
    if hasattr(model.model, 'layers'): return model.model.layers
    raise AttributeError('Cannot locate transformer layers')

def _tokenize(text, device):
    return tokenizer(text, return_tensors='pt', truncation=True).input_ids.to(device)


# ============================================================
# DATA GENERATION -- single NiH examples (not pairs)
# ============================================================

def make_nih_example(num_docs=5, needle_pos=None, seed=42):
    """
    Generate a single NiH example.
    needle_pos: where the needle appears (0=first doc, num_docs-1=last doc).
    Returns (text, answer, obj_name)
    """
    rng = random.Random(seed)
    obj = rng.choice(_objects)
    val = rng.choice(_answers)
    needle = f'The special {obj} is {val}.'
    dists = [f'The capital of {c} is {cy}.' for c, cy in rng.sample(_countries, num_docs)]
    if needle_pos is None:
        needle_pos = num_docs // 2  # middle = hardest
    docs = dists[:needle_pos] + [needle] + dists[needle_pos:]
    text = ' | '.join(f'Doc {i+1}: {d}' for i, d in enumerate(docs))
    text += f'\nRetrieve the {obj} from the docs above.\nAnswer: '
    return text, val, obj


# ============================================================
# PART A: ACTIVATION PATCHING -- Corrupted-Input Design
# ============================================================
print('\n' + '=' * 60)
print('PART A: Activation Patching -- Causal Validation')
print('=' * 60)

# Principle: same text, remove needle -> corrupted, patch clean act -> should recover

# --- Activation cache ---
class _ActCache:
    def __init__(self): self.activations = {}; self.hooks = []
    def clear(self):
        self.activations = {}
        for h in self.hooks: h.remove()
        self.hooks = []

# --- Bottleneck from ERB analysis ---
BOTTLENECK_LAYER = 31  # last layer = most congested across all models

# --- Generate clean/corrupted pairs ---
# Clean: full NiH text with needle at middle position
# Corrupted: same text but needle replaced with a neutral distractor doc
eval_data_clean = []
eval_data_corrupted = []
for s in range(42, 72):  # 30 pairs
    text, val, obj = make_nih_example(num_docs=5, needle_pos=2, seed=s)
    # Corrupted: replace the needle with another distractor
    rng2 = random.Random(s + 1000)
    dummy_country = rng2.choice(_countries)
    corrupted_text = text.replace(
        f'The special {obj} is {val}.',
        f'The capital of {dummy_country[0]} is {dummy_country[1]}.'
    )
    eval_data_clean.append((text, val))
    eval_data_corrupted.append((corrupted_text, val))

print(f'Generated {len(eval_data_clean)} clean/corrupted NiH pairs (needle at pos 2 of 5)')

# --- Baseline scores: clean vs corrupted ---
print('\nComputing baseline scores...')
scores_clean = [score_answer(model, tokenizer, text, val, model_device)
                for text, val in eval_data_clean]
scores_corrupted = [score_answer(model, tokenizer, text, val, model_device)
                    for text, val in eval_data_corrupted]

mean_clean = np.mean(scores_clean)
mean_corrupted = np.mean(scores_corrupted)
print(f'  Clean (has needle):       {mean_clean:.4f}')
print(f'  Corrupted (no needle):    {mean_corrupted:.4f}')
print(f'  Drop from removing needle: {mean_clean - mean_corrupted:.4f}')

# --- Patch: inject clean residual states (which encode the needle) into corrupted run ---
print(f'\nPatching: Inject clean Layer {BOTTLENECK_LAYER} activations into corrupted run...')
print(f'Hypothesis: residual states at bottleneck carry needle info,')
print(f'  patching them into corrupted run should partially restore prediction.')

patched_scores = []
for i, ((ctext, val), (txtext, _)) in enumerate(zip(eval_data_clean, eval_data_corrupted)):
    ids_clean = _tokenize(ctext, model_device)
    ids_corrupted = _tokenize(txtext, model_device)

    # Step 1: Cache clean activations at bottleneck layer
    cache = _ActCache()
    layers = _get_layers(model)
    for li in [BOTTLENECK_LAYER]:
        if li < len(layers):
            def make_hook(lidx):
                def hook(m, i, o):
                    out = o[0] if isinstance(o, tuple) else o
                    cache.activations[f'L{lidx}'] = out.detach().clone()
                return hook
            cache.hooks.append(layers[li].register_forward_hook(make_hook(li)))
    with torch.no_grad():
        _ = model(ids_clean)
    clean_acts = {k: v for k, v in cache.activations.items()}
    cache.clear()

    # Step 2: Run corrupted input with clean activation patch
    # Patch ALL positions at the bottleneck layer (full seq replacement)
    def make_patch_hook(lidx, source_acts):
        def hook(m, i, o):
            out = o[0] if isinstance(o, tuple) else o
            src = source_acts.get(f'L{lidx}')
            if src is None: return o
            min_len = min(out.shape[1], src.shape[1])
            patched = out.clone()
            patched[0, :min_len, :] = src[0, :min_len, :]
            return (patched,) + o[1:] if isinstance(o, tuple) else patched
        return hook

    for li in [BOTTLENECK_LAYER]:
        if li < len(layers):
            cache.hooks.append(layers[li].register_forward_hook(make_patch_hook(li, clean_acts)))

    with torch.no_grad():
        logits = model(ids_corrupted).logits
    cache.clear()

    log_probs = torch.log_softmax(logits[0, -1, :], dim=-1)
    ans_tokens = tokenizer.encode(' ' + val, add_special_tokens=False)
    if not ans_tokens: ans_tokens = tokenizer.encode(val, add_special_tokens=False)
    patched_scores.append(log_probs[ans_tokens[0]].item() if ans_tokens else 0.0)

    if (i + 1) % 10 == 0:
        print(f'  [{i+1}/{len(eval_data_clean)}] Running mean patched={np.mean(patched_scores):.4f}')

mean_patched = np.mean(patched_scores)
gap = max(mean_clean - mean_corrupted, 0.001)
recovery = (mean_patched - mean_corrupted) / gap

print(f'\n' + '=' * 60)
print('PATCHING RESULTS (corrupted-input design)')
print('=' * 60)
print(f'  Clean (has needle):         {mean_clean:.4f}')
print(f'  Corrupted (no needle):      {mean_corrupted:.4f}')
print(f'  Patched (L31 clean act):    {mean_patched:.4f}')
print(f'  Gap (clean - corrupted):    {gap:.4f}')
print(f'  Recovery rate:              {recovery:.2%}')
if recovery > 0:
    print(f'  Causal evidence: residual stream at Layer 31 carries needle information')
else:
    print(f'  Note: a negative/zero recovery means patching at L31 alone is insufficient;')
    print(f'    information may be distributed across multiple layers.')

# Save
os.makedirs('experiments', exist_ok=True)
patching_results = {
    'design': 'corrupted-input (same text, needle removed)',
    'mean_clean': float(mean_clean),
    'mean_corrupted': float(mean_corrupted),
    'mean_patched': float(mean_patched),
    'gap': float(gap),
    'recovery_rate': float(recovery),
    'n_examples': len(eval_data_clean),
    'bottleneck_layer': BOTTLENECK_LAYER,
}
with open('experiments/patching_results.json', 'w') as f:
    json.dump(patching_results, f, indent=2)
print(f'Results saved to experiments/patching_results.json')


# ============================================================
# PART B: INFORMATION HIGHWAY -- Fixed Training
# ============================================================
print('\n' + '=' * 60)
print('PART B: Information Highway -- Training & Accuracy Evaluation')
print('=' * 60)

# Build bypass at congested layers
bypass_width = head_dim * 2
BYPASS_LAYERS = [0, 16, 31]  # most congested layers across all models

class _HighwayBypass(nn.Module):
    """Low-rank residual bypass with Xavier init for stable training."""
    def __init__(self, h_size, b_width):
        super().__init__()
        self.down = nn.Linear(h_size, b_width, bias=False)
        self.up = nn.Linear(b_width, h_size, bias=False)
        self.gate = nn.Parameter(torch.zeros(1))
        # Xavier init (small gain) -- FIX: was zero-init
        nn.init.xavier_uniform_(self.down.weight, gain=0.01)
        nn.init.xavier_uniform_(self.up.weight, gain=0.01)
    def forward(self, x):
        return torch.sigmoid(self.gate) * self.up(self.down(x))

bypasses = nn.ModuleDict()
for layer in BYPASS_LAYERS:
    bypasses[f'bypass_{layer}'] = _HighwayBypass(hidden_size, bypass_width)
bypasses = bypasses.to(device=model_device, dtype=torch.float32)

trainable_params = sum(p.numel() for p in bypasses.parameters())
pct_of_backbone = 100 * trainable_params / backbone_params
print(f'Bypass layers: {BYPASS_LAYERS}, width={bypass_width}, dtype=float32')
print(f'Trainable: {trainable_params:,} params ({pct_of_backbone:.4f}% of backbone)')

# --- Bypass forward helper ---
def _forward_with_bypass(model, bypasses, input_ids, bypass_layers):
    hooks = []
    layers = _get_layers(model)
    for li in bypass_layers:
        if li < len(layers):
            def make_hook(lidx):
                def hook(m, i, o):
                    bk = f'bypass_{lidx}'
                    if bk not in bypasses: return o
                    hidden = o[0] if isinstance(o, tuple) else o
                    hidden_fp32 = hidden.float()
                    bypass_out = bypasses[bk](hidden_fp32)
                    result = hidden + bypass_out.to(hidden.dtype)
                    return (result,) + o[1:] if isinstance(o, tuple) else result
                return hook
            hooks.append(layers[li].register_forward_hook(make_hook(li)))
    with torch.no_grad():
        logits = model(input_ids).logits
    for h in hooks: h.remove()
    return logits

def score_with_bypass(model, bypasses, tokenizer, text, answer, device, blayers):
    inputs = _tokenize(text, device)
    logits = _forward_with_bypass(model, bypasses, inputs, blayers)
    log_probs = torch.log_softmax(logits[0, -1, :], dim=-1)
    ans_tokens = tokenizer.encode(' ' + answer, add_special_tokens=False)
    if not ans_tokens: ans_tokens = tokenizer.encode(answer, add_special_tokens=False)
    return log_probs[ans_tokens[0]].item() if ans_tokens else 0.0

# --- Pre-training evaluation ---
print('\nPre-training accuracy check...')
bypass_untrained_scores = [score_with_bypass(model, bypasses, tokenizer, ct, val, model_device, BYPASS_LAYERS)
                           for ct, val in eval_data_corrupted]
mean_bypass_untrained = np.mean(bypass_untrained_scores)
print(f'  Corrupted (no bypass):   {mean_corrupted:.4f}')
print(f'  Corrupted + bypass init: {mean_bypass_untrained:.4f}')
print(f'  Clean (has needle):      {mean_clean:.4f}')

# --- Training loop ---
print(f'\nTraining on 200 NiH examples (5 epochs)...')

# Generate training data
train_examples = [make_nih_example(num_docs=5, needle_pos=2, seed=s) for s in range(200, 400)]

optimizer = torch.optim.AdamW(bypasses.parameters(), lr=1e-3)
history = {'epoch': [], 'loss': [], 'corrupted_score': [], 'recovery': []}

for epoch in range(5):
    epoch_losses = []
    random.shuffle(train_examples)

    for batch_start in range(0, len(train_examples), 8):
        batch = train_examples[batch_start:batch_start + 8]
        batch_loss = 0.0

        for text, val, _ in batch:
            ids = _tokenize(text, model_device)
            # Labels: only predict the last token (answer), mask the rest
            labels = ids.clone()
            labels[:, :-1] = -100

            # Forward with bypass hooks
            hooks = []
            layers = _get_layers(model)
            for li in BYPASS_LAYERS:
                if li < len(layers):
                    def make_thook(lidx):
                        def thook(m, i, o):
                            bk = f'bypass_{lidx}'
                            if bk not in bypasses: return o
                            hidden = o[0] if isinstance(o, tuple) else o
                            hidden_fp32 = hidden.float()
                            bypass_out = bypasses[bk](hidden_fp32)
                            result = hidden + bypass_out.to(hidden.dtype)
                            return (result,) + o[1:] if isinstance(o, tuple) else result
                        return thook
                    hooks.append(layers[li].register_forward_hook(make_thook(li)))

            optimizer.zero_grad()
            outputs = model(ids, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(bypasses.parameters(), 1.0)
            optimizer.step()

            for h in hooks: h.remove()
            batch_loss += loss.item()

        epoch_losses.append(batch_loss / len(batch))

    # Evaluate after each epoch
    bypass_scores = [score_with_bypass(model, bypasses, tokenizer, ct, val, model_device, BYPASS_LAYERS)
                     for ct, val in eval_data_corrupted]
    mean_bs = np.mean(bypass_scores)
    epoch_rec = (mean_bs - mean_corrupted) / max(mean_clean - mean_corrupted, 0.001)

    epoch_loss = float(np.mean(epoch_losses))
    history['epoch'].append(epoch + 1)
    history['loss'].append(epoch_loss)
    history['corrupted_score'].append(float(mean_bs))
    history['recovery'].append(float(epoch_rec))

    print(f'  Epoch {epoch+1}: loss={epoch_loss:.4f}, score={mean_bs:.4f} (clean={mean_clean:.4f}), recovery={epoch_rec:.2%}')

    # Early stopping
    if epoch >= 2 and abs(history['loss'][-1] - history['loss'][-2]) < 0.0001:
        print(f'  Loss plateaued, stopping early')
        break

# --- Final evaluation ---
bypasses.eval()
bypass_trained_scores = [score_with_bypass(model, bypasses, tokenizer, ct, val, model_device, BYPASS_LAYERS)
                         for ct, val in eval_data_corrupted]
mean_bypass_trained = np.mean(bypass_trained_scores)
final_recovery = (mean_bypass_trained - mean_corrupted) / max(mean_clean - mean_corrupted, 0.001)

print(f'\n' + '=' * 60)
print('INFORMATION HIGHWAY RESULTS')
print('=' * 60)
print(f'  Clean (has needle):              {mean_clean:.4f}')
print(f'  Corrupted (no needle):           {mean_corrupted:.4f}')
print(f'  Corrupted + bypass (untrained):  {mean_bypass_untrained:.4f}')
print(f'  Corrupted + bypass (trained):    {mean_bypass_trained:.4f}')
print(f'  Improvement over corrupted:      {mean_bypass_trained - mean_corrupted:+.4f}')
print(f'  Recovery rate:                   {final_recovery:.2%}')
print(f'  Bypass params:                   {trainable_params:,} ({pct_of_backbone:.4f}%)')
print(f'  Layers bypassed:                 {BYPASS_LAYERS}')
if final_recovery > 0:
    print(f'  The bypass learns to route needle information through the congestion bottleneck')

# Save
highway_results = {
    'design': 'xavier_init + answer-only labels + fp32 bypass',
    'mean_clean': float(mean_clean),
    'mean_corrupted': float(mean_corrupted),
    'mean_bypass_untrained': float(mean_bypass_untrained),
    'mean_bypass_trained': float(mean_bypass_trained),
    'recovery_rate': float(final_recovery),
    'improvement': float(mean_bypass_trained - mean_corrupted),
    'bypass_layers': BYPASS_LAYERS,
    'bypass_width': bypass_width,
    'trainable_params': trainable_params,
    'backbone_params': backbone_params,
    'pct_of_backbone': pct_of_backbone,
    'bypass_dtype': 'float32',
    'training_history': history,
}
with open('experiments/highway_results.json', 'w') as f:
    json.dump(highway_results, f, indent=2)
print(f'Results saved to experiments/highway_results.json')

torch.save(bypasses.state_dict(), 'experiments/highway_bypass.pt')
print(f'Bypass weights saved to experiments/highway_bypass.pt')

print('\nStage 3 complete: Activation Patching + Information Highway')


In [ ]:
!pip install accelerate -q

In [ ]:
%pip install bitsandbytes

## 6. Stage 4: Ablation Studies

In [6]:
# === Stage 4: Completely self-contained ablation studies ===
# Works standalone after Environment Setup (cells 1-3).

import torch, torch.nn as nn, random
import numpy as np

print('=' * 60)
print('STAGE 4: Ablation Studies')
print('=' * 60)

# ===== Self-contained: model loading =====
MODEL_NAME = 'EleutherAI/pythia-2.8b'
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    import bitsandbytes as bnb
    from transformers import BitsAndBytesConfig
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=qcfg,
        device_map='auto', trust_remote_code=True)
except ImportError:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
        torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
        device_map='auto' if device == 'cuda' else None,
        trust_remote_code=True)
    if device == 'cuda': model = model.to(device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model.eval()
cfg = model.config
num_layers = next(getattr(cfg, a) for a in ['num_hidden_layers','n_layer','num_layers'] if hasattr(cfg,a))
hidden_size = cfg.hidden_size
head_dim = getattr(cfg, 'head_dim', hidden_size // cfg.num_attention_heads)
model_device = next(model.parameters()).device
model_dtype = next(model.parameters()).dtype
backbone_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {num_layers} layers, {hidden_size} hidden, {backbone_params:,} params')
print(f'Device: {model_device}, dtype: {model_dtype}')

# ===== Self-contained: data generation + ERB =====
class _ERB:
    def __init__(self, eps=1e-8):
        self.eps = eps
    def _norm(self, x):
        return torch.norm(x, dim=-1, p=2)
    def compute_all(self, activations, nlayers):
        fk = [k for k in activations if k.endswith('_input')][0]
        S = activations[fk].shape[1]
        E = torch.zeros(nlayers, S)
        for i in range(nlayers):
            ki, ko = f'layer_{i}_input', f'layer_{i}'
            if ki not in activations or ko not in activations: continue
            li, lo = activations[ki], activations[ko]
            if li.ndim == 3: li, lo = li.mean(0), lo.mean(0)
            E[i] = self._norm(lo) / (self._norm(lo - li) + self.eps)
        return E, None

class _ActCache:
    def __init__(self): self.activations = {}; self.hooks = []
    def reset(self): self.activations = {}; self.hooks = []
    def _hook(self, n, si):
        def h(m, i, o):
            if si: self.activations[f'{n}_input'] = i[0].detach().cpu()
            self.activations[n] = o[0].detach().cpu()
        return h
    def clear(self):
        self.reset()
        for hk in self.hooks: hk.remove()
        self.hooks = []

def _run_erb(input_ids, model, nlayers, eps):
    """Run forward pass and compute ERB."""
    cache = _ActCache()
    layers = (model.gpt_neox.layers if hasattr(model, 'gpt_neox') else
              model.model.layers if hasattr(model.model, 'layers') else
              model.transformer.h)
    for idx in range(min(nlayers, len(layers))):
        cache.hooks.append(layers[idx].register_forward_hook(cache._hook(f'layer_{idx}', True)))
    with torch.no_grad():
        outputs = model(input_ids)
    for hk in cache.hooks: hk.remove()
    erb = _ERB(eps)
    m, _ = erb.compute_all(cache.activations, nlayers)
    return m.mean().item()

# ===== Self-contained: NiH data =====
_countries = [('France','Paris'),('Germany','Berlin'),('Italy','Rome'),('Spain','Madrid'),
              ('UK','London'),('Japan','Tokyo'),('China','Beijing'),('India','New Delhi')]

def _gen_nih_examples(num=5, docs=5, seed=42):
    rng = random.Random(seed)
    objs = ['planet','star','color','element','animal','fruit']
    examples = []
    for _ in range(num):
        obj = rng.choice(objs); val = rng.choice(['A','B','C','D','E'])
        needle = f'The special {obj} is {val}.'
        dists = [f'The capital of {c} is {cy}.' for c,cy in rng.sample(_countries, docs)]
        rng.shuffle(dists)
        text = ' | '.join(f'Doc {i+1}: {d}' for i,d in enumerate(dists)) + f'\nRetrieve the {obj}.\nAnswer: '
        examples.append(text)
    return examples

# ===== Ablation 1: ERB Epsilon Sensitivity =====
print('\n--- Ablation 1: ERB Epsilon Sensitivity ---')
examples = _gen_nih_examples(num=3)
for eps in [1e-6, 1e-8, 1e-10]:
    vals = []
    for text in examples:
        ids = tokenizer(text, return_tensors='pt', truncation=True).input_ids.to(model_device)
        m = _run_erb(ids, model, num_layers, eps)
        vals.append(m)
    print(f'  eps={eps:.0e}: mean ERB = {np.mean(vals):.5f} ± {np.std(vals):.5f}')

# ===== Ablation 2: Highway Width Analysis =====
print('\n--- Ablation 2: Highway Width Analysis ---')
class _ByPass(nn.Module):
    def __init__(self, hs, bw):
        super().__init__()
        self.down = nn.Linear(hs, bw, bias=False)
        self.up = nn.Linear(bw, hs, bias=False)
        self.gate = nn.Parameter(torch.zeros(1))
        nn.init.zeros_(self.down.weight); nn.init.zeros_(self.up.weight)

for wf in [1, 2, 4]:
    bw = head_dim * wf
    bypass = _ByPass(hidden_size, bw)
    n_p = sum(p.numel() for p in bypass.parameters())
    pct = 100 * n_p / max(backbone_params, 1)
    print(f'  Width x{wf} (dim={bw}): {n_p:,} params ({pct:.4f}% of backbone)')

print('\nStage 4 complete!')

STAGE 4: Ablation Studies


Loading weights: 100%|██████████| 388/388 [00:02<00:00, 172.56it/s, Materializing param=gpt_neox.layers.31.post_attention_layernorm.weight] 


Model loaded: 32 layers, 2560 hidden, 1,516,917,760 params
Device: cuda:0, dtype: torch.float16

--- Ablation 1: ERB Epsilon Sensitivity ---
  eps=1e-06: mean ERB = 5.34931 ± 0.01154
  eps=1e-08: mean ERB = 5.34931 ± 0.01154
  eps=1e-10: mean ERB = 5.34931 ± 0.01154

--- Ablation 2: Highway Width Analysis ---
  Width x1 (dim=80): 409,601 params (0.0270% of backbone)
  Width x2 (dim=160): 819,201 params (0.0540% of backbone)
  Width x4 (dim=320): 1,638,401 params (0.1080% of backbone)

Stage 4 complete!


## 7. Results Summary & Download

In [7]:
print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)

# Load and display congestion mapping results
results_path = 'experiments/congestion_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print(f'\n=== CONGESTION MAPPING ===')
    print(f'Total experiment runs: {len(results)}')
    nih_runs = [(k, v) for k, v in results.items() if v.get('task_type', 'nih') == 'nih']
    mqa_runs = [(k, v) for k, v in results.items() if v.get('task_type') == 'multidoc_qa']
    print(f'  NiH runs: {len(nih_runs)}')
    print(f'  Multi-doc QA runs: {len(mqa_runs)}')
    for label, runs in [('NiH', nih_runs), ('MultiDoc QA', mqa_runs)]:
        rhos = [v['spearman_rho'] for _, v in runs if 'spearman_rho' in v]
        if rhos:
            print(f'  {label}: mean ρ = {np.mean(rhos):.3f} ± {np.std(rhos):.3f}')
    # Bottleneck summary
    all_bn = []
    for _, v in results.items():
        if 'bottlenecks' in v: all_bn.extend(v['bottlenecks'])
    if all_bn:
        from collections import Counter
        layers = Counter(b[0] for b in all_bn)
        print(f'  Most congested layers: {layers.most_common(5)}')

# Load patching results
patching_path = 'experiments/patching_results.json'
if os.path.exists(patching_path):
    with open(patching_path) as f:
        pr = json.load(f)
    print(f'\n=== ACTIVATION PATCHING ===')
    print(f'  Easy (end):     {pr["mean_easy"]:.4f}')
    print(f'  Hard (middle):  {pr["mean_hard"]:.4f}')
    print(f'  Patched:        {pr["mean_patched"]:.4f}')
    print(f'  Recovery rate:  {pr["recovery_rate"]:.2%}')
    print(f'  Bottleneck:     Layer {pr["bottleneck_layer"]} pos {pr["patch_positions"]}')

# Load highway results
highway_path = 'experiments/highway_results.json'
if os.path.exists(highway_path):
    with open(highway_path) as f:
        hr = json.load(f)
    print(f'\n=== INFORMATION HIGHWAY ===')
    print(f'  Easy (end):               {hr["mean_easy"]:.4f}')
    print(f'  Hard (middle):            {hr["mean_hard"]:.4f}')
    print(f'  Bypass untrained:         {hr["mean_bypass_untrained"]:.4f}')
    print(f'  Bypass trained:           {hr["mean_bypass_trained"]:.4f}')
    print(f'  Recovery rate:            {hr["recovery_rate"]:.2%}')
    print(f'  Bypass params:            {hr["trainable_params"]:,} ({hr["pct_of_backbone"]:.4f}% of backbone)')
    print(f'  Bypass layers:            {hr["bypass_layers"]}')

# Load ablation results
ablation_path = 'experiments/stage4_ablation.json'
if os.path.exists(ablation_path):
    with open(ablation_path) as f:
        ar = json.load(f)
    print(f'\n=== ABLATION STUDIES ===')
    if 'erb_epsilon' in ar:
        print(f'  ERB ε sensitivity:')
        for eps, vals in ar['erb_epsilon'].items():
            print(f'    ε={eps}: mean={vals["mean_erb"]:.5f}')
    if 'highway_width' in ar:
        print(f'  Highway width analysis:')
        for w, vals in ar['highway_width'].items():
            print(f'    {w}: {vals["trainable_params"]:,} params ({vals["pct_of_backbone"]:.4f}%)')

print('\nAll stages complete!')

RESULTS SUMMARY

=== ACTIVATION PATCHING ===


KeyError: 'mean_easy'

In [ ]:
# List all output files
for root, dirs, files in os.walk('experiments'):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'  {path} ({size:,} bytes)')
for root, dirs, files in os.walk('figures'):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f'  {path} ({size:,} bytes)')

In [ ]:
# Download results as zip
import os, glob

# Collect all result files
files_to_zip = []
for pattern in ['experiments/*', 'figures/*', 'config.yaml']:
    matches = glob.glob(pattern)
    files_to_zip.extend([f for f in matches if os.path.isfile(f)])

if files_to_zip:
    # Print file list
    print('Files to download:')
    for f in sorted(files_to_zip):
        size = os.path.getsize(f)
        print(f'  {f} ({size:,} bytes)')
    
    file_list = ' '.join(files_to_zip)
    !zip -r experimental_results.zip {file_list}
    from IPython.display import FileLink
    display(FileLink('experimental_results.zip'))
    print(f'\nZipped {len(files_to_zip)} files -> experimental_results.zip')
else:
    print('No result files found to zip.')
    print('Run at least Stage 1-3 first.')
    !ls -la

## Notes

- **Pythia-6.9B**: Uncomment in the config cell and set `load_in_8bit: True`. Requires ~14 GB VRAM.
- **LLaMA-7B**: Same as above. Some LLaMA variants may need auth token (`huggingface-cli login`).
- **Training**: Stage 3 provides a demo. For full training (1000+ examples, 5+ epochs), increase `training_examples` and `epochs` in config.
- **OOM issues**: Reduce `examples_per_setting` or `context_lengths`.
- **Existing results**: If you have existing results from CPU runs, upload them to the `experiments/` folder before running.